# Integrity checks

The routines in this notebook perform basic checks on the selected datasets that are easier to be made programmatic:
- checking for NaNs
- checking for aphysical quantities
- dimensions of datasets are what we expect them to be

Eventually it could be useful to operationalize these tests into the production system to make it automated. For now we'll have a section that performs integrity checks on the input datasets and one evaluates output datasets.

In [67]:
%load_ext autoreload
%autoreload 2
import duckdb
import geopandas as gpd
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterix
import seaborn as sns
import xarray as xr

from rasterix.rasterize.rasterio import geometry_clip
import pint
from srm import catalog
from srm.utils import icechunk_store_to_dataset, clean_up_dataset
from srm.qaqc import check_physical_constraints, confirm_coords

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
catalog

Dataset Catalog (4 datasets)
--------------------------------------------------------------------------------
CESM-WACCM-Historical-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/CESM2-WACCM-Historical/icechunk/icechunk
CESM-WACCM-G6-1.5K-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/CESM-WACCM-G6-1.5K/icechunk/icechunk
CESM2-WACCM-SSP245-icechunk | icechunk   | s3://carbonplan-srm/input/tensor/CESM2-WACCM-SSP245/icechunk/icechunk
ERA5                 | icechunk   | s3://carbonplan-srm/input/tensor/era5_rechunked_resampled.icechunk

# Get input data

Once the input datasets are in their final place for use and are all done with any pre-processing (e.g. rechunking), they won't change. So, we only have to run the integrity routines once and they can live in their own separate notebook.

### Output from three climate model simulations

These simulations were conducted in the Community Earth System Model version 2 -- Whole Atmosphere Community Climate Model (CESM2-WACCM)
* Historical (1978 - 2014)
* SSP245 (2015 - 2070)
* G6-1.5K (2035 - 2085)

In [5]:
gcm = "CESM2-WACCM"
scenario = "SSP245" # "G6-1.5K" , "Historical"
variables = ['TASMEAN', 'TASMIN', 'TASMAX', 'PREC', 'RSDS']
dataset_name = f"{gcm}-{scenario}-icechunk"

In [21]:
ds = icechunk_store_to_dataset(dataset_name, catalog)

In [23]:
ds = clean_up_dataset(ds, gcm)[variables]

### Observations: ERA5

In [8]:
era5 = icechunk_store_to_dataset("ERA5", catalog)

In [37]:
# @claire what does the rasterix.assign_index do? why do we only do it for era5 and not the GCM datasets?
era5 = clean_up_dataset(era5, "ERA5")[variables].pipe(rasterix.assign_index)

In [35]:
era5 = icechunk_store_to_dataset("ERA5", catalog)

### Integrity checks

In [81]:
# test running on a 2x2 box for expediency since the dataset is chunked along full time dimension
# and so would take a long time to check fully
out = ds.sel(latitude=slice(46, 48), 
                  longitude=slice(-124, -122.0)).sel(ensemble_member='006').load()

IcechunkError:   x error streaming bytes from object store streaming error
  | 
  | context:
  |    0: icechunk::storage::s3::get_object_range_buf
  |            with settings=Settings { concurrency: None, retries: None, unsafe_use_conditional_update: None, unsafe_use_conditional_create: None, unsafe_use_metadata: None, storage_class: None,
  | metadata_storage_class: None, chunks_storage_class: None, minimum_size_for_multipart_upload: None } key="input/tensor/CESM2-WACCM-SSP245/icechunk/icechunk/chunks/ZF8VACXXGB2WQFHDKYZ0"
  | range=71645789..83586753
  |              at icechunk/src/storage/s3.rs:984
  |    1: icechunk::storage::s3::fetch_chunk
  |            with id=ZF8VACXXGB2WQFHDKYZ0 range=0..95527717
  |              at icechunk/src/storage/s3.rs:660
  |    2: icechunk::asset_manager::fetch_chunk
  |            with chunk_id=ZF8VACXXGB2WQFHDKYZ0 range=0..95527717
  |              at icechunk/src/asset_manager.rs:361
  |    3: icechunk::store::get
  |            with key="TREFHT

In [68]:
check_physical_constraints(out)

Number of negative precipitation values: 0
Number of outlandishly high precipitation values: 0
Number of outlandishly high TASMEAN values: 0
Number of outlandishly high TASMAX values: 0
Number of outlandishly high TASMIN values: 0
Number of outlandishly low TASMEAN values: 0
Number of outlandishly low TASMAX values: 0
Number of outlandishly low TASMIN values: 0
Number of times TASMIN exceeds TASMEAN: 0
Number of times TASMEAN exceeds TASMAX: 0
Number of times TASMIN exceeds TASMAX: 0


In [ ]:
check_physical_constraints(era5.sel(latitude=slice(47, 48), 
                  longitude=slice(-122, -121)))

In [85]:
era5.sel(latitude=slice(48, 47), 
        longitude=slice(-122, -121))

<xarray.Dataset> Size: 493GB
Dimensions:      (time: 23741, latitude: 721, longitude: 1440)
Coordinates:
  * time         (time) datetime64[ns] 190kB 1950-01-01 ... 2014-12-31
  * latitude     (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude    (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
  * spatial_ref  int64 8B 0
Data variables:
    TASMEAN      (time, latitude, longitude) float32 99GB dask.array<chunksize=(730, 144, 288), meta=np.ndarray>
    TASMIN       (time, latitude, longitude) float32 99GB dask.array<chunksize=(730, 144, 288), meta=np.ndarray>
    TASMAX       (time, latitude, longitude) float32 99GB dask.array<chunksize=(730, 144, 288), meta=np.ndarray>
    PREC         (time, latitude, longitude) float32 99GB dask.array<chunksize=(730, 144, 288), meta=np.ndarray>
    RSDS         (time, latitude, longitude) float32 99GB dask.array<chunksize=(730, 144, 288), meta=np.ndarray>
Indexes:
    spatial_ref  CRSIndex (crs=EPSG:4326)
  ┌ longitude    RasterIndex (crs=EPSG:4326)
  └ latitude
Attributes:
    last_updated:           2025-09-15 01:44:32.188024+00:00
    valid_time_start:       1950-01-01
    valid_time_stop:        2014-12-31
    valid_time_stop_era5t:  2014-12-31

In [72]:
confirm_coords(era5)

'Latitude and longitude match expectation'